## Multi-Tool Calling with MAF and Microsoft Foundry

**Fundamental logic:** A multi-tool agent lets the model choose among several capabilities according to the user?s request.


![mutliple-tools](./Assets/multiple-tools.png)

**Fundamental logic:** The diagram shows one agent routing work to either hosted code execution or the Microsoft Learn MCP server.


In [1]:
# Fundamental logic: Pinned dependencies make the example reproducible against the APIs used in the remaining cells.

%pip install agent-framework==1.0.0b251209 python-dotenv azure-ai-projects==2.0.0b2

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Setting Up the Environment

**Fundamental logic:** The environment cell supplies the project endpoint and model deployment used to create the Foundry agent.


In [2]:
# Fundamental logic: Loading values from .env keeps deployment configuration out of notebook logic and avoids hard-
# coded project settings.

import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential

load_dotenv()
project_endpoint = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
model = os.getenv("AI_FOUNDRY_DEPLOYMENT_NAME")

print("Project Endpoint: ", project_endpoint)
print("Model: ", model)

Project Endpoint:  https://ajay-agent-project111-resource.services.ai.azure.com/api/projects/ajay-agent-project111
Model:  ajay-gpt-4o


### Loading the Code Interpreter Tool

**Fundamental logic:** The code interpreter gives the agent a sandboxed runtime for calculations and Python execution.


In [3]:
# Fundamental logic: Creating the tool object makes code execution available for attachment; the model decides when a
# request benefits from it.

from agent_framework import AgentRunResponse, ChatResponseUpdate, HostedCodeInterpreterTool

code_interpreter_tool = HostedCodeInterpreterTool()

### Loading the Hosted MCP Tool

**Fundamental logic:** The MCP tool connects the same agent to an external source of Microsoft Learn capabilities and documentation.


In [4]:
# Fundamental logic: This hosted tool points at an MCP server whose advertised operations can be selected by the
# model.

from agent_framework import HostedMCPTool

ms_learn_mcp_tool = HostedMCPTool(
    name = "Microsoft Learn MCP Tool",
    url = "https://learn.microsoft.com/api/mcp"
)

### Creating our Multi-Tool Agent

**Fundamental logic:** The next cell creates one conversation-scoped agent and attaches both tools, enabling model-driven tool selection.


In [5]:
# Fundamental logic: The agent receives a list of tools. Its model reasons about the prompt and invokes the most
# suitable capability.

from agent_framework.azure import AzureAIClient
from azure.identity.aio import AzureCliCredential
from azure.ai.projects.aio import AIProjectClient

async def create_multi_tool_agent():
    credential = AzureCliCredential()
    
    # creating the Foundry Project Client
    project_client = AIProjectClient(
        endpoint=project_endpoint,
        credential=credential
    )

    # creating a conversation using the OpenAI Client
    openai_client = project_client.get_openai_client()
    conversation = await openai_client.conversations.create()
    conversation_id = conversation.id
    print("Conversation ID: ", conversation_id)
    
    # creating the Azure AI Client to interact with the Agent in Foundry
    agent_client = AzureAIClient(
        project_client = project_client,
        conversation_id = conversation_id,
        model_deployment_name=model
    )
    
    # creating an agent in Foundry
    agent = agent_client.create_agent(
        name="multi-tool-agent",
        instructions="You are a helpful multi-tool agent",
        tools = [code_interpreter_tool, ms_learn_mcp_tool]
    )
    return agent, credential, agent_client

agent, credential, agent_client = await create_multi_tool_agent()

Conversation ID:  conv_e6ba361bc36810a500zialzEZqTiszmesn67KbIbMDGinjxf8W


### Creating an Async Function to Handle User Approvals

**Fundamental logic:** Some tools request approval before execution. The helper repeatedly responds to those requests until the run finishes.


In [6]:
# Fundamental logic: This sample auto-approves every requested action. In production, approvals should be reviewed or
# governed by policy.

from agent_framework import ChatMessage

async def handle_approvals_with_thread(query: str, agent):
    result = await agent.run(query)
    while len(result.user_input_requests) > 0:
        new_input = []
        for user_input_needed in result.user_input_requests:
            new_input.append(
                ChatMessage(
                    role="user",
                    contents=[user_input_needed.create_response(True)],
                )
            )
        result = await agent.run(new_input)
    return result

### Running Different Queries on the Agent

**Fundamental logic:** The next queries demonstrate that the same agent can route different tasks to different attached tools.


In [7]:
# Fundamental logic: This computational prompt should lead the agent toward the hosted code interpreter.

query = "Can you write a Python function to calculate the factorial of a number using recursion and give me the output for number 100"
response = await handle_approvals_with_thread(query, agent)

print("\n🧠 Assistant:\n", response)


🧠 Assistant:
 def factorial_recursive(n):
    """
    Calculate the factorial of a number using recursion.

    Args:
    n (int): Number to calculate factorial for.

    Returns:
    int: Factorial of the number.
    """
    if n == 0 or n == 1:
        return 1
    return n * factorial_recursive(n - 1)

# Calculate factorial of 100
factorial_100 = factorial_recursive(100)
factorial_100 The factorial of the number 100 is:

93326215443944152681699238856266700490715968264381621468592963895217599993229915608941463976156518286253697920827223758251185210916864000000000000000000000000


In [8]:
# Fundamental logic: This documentation prompt should lead the agent toward the Microsoft Learn MCP tool; a new
# thread can isolate conversation context.

thread = agent.get_new_thread()
query = "How to create an Azure storage account using az cli?"
response = await handle_approvals_with_thread(query, agent)

print("\n🧠 Assistant:\n", response)


🧠 Assistant:
 Here is a basic Azure CLI script to create an Azure storage account:

```bash
az storage account create \
        --name "<storage-account-name>" \
        --resource-group "<resource-group-name>" \
        --location "<location>" \
        --sku Standard_LRS \
        --kind StorageV2
```

### Parameters:
- **`<storage-account-name>`**: Replace with your desired unique name for the storage account.
- **`<resource-group-name>`**: Specify the resource group where the account will be created.
- **`<location>`**: Specify the Azure region where the account will be hosted (e.g., `eastus`, `westus`).

For more details, you can access the [official documentation](https://learn.microsoft.com/azure/container-apps/tutorial-event-driven-jobs#set-up-a-storage-queue).
